In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/data.csv")
df.head()

In [ ]:
df.columns
df.dtypes
df.shape
df.info()
df.describe()

In [ ]:
df.duplicated().sum()
df.isna().sum
df[df["CREDIT_LIMIT"].isna()]
df = df.dropna(subset=["CREDIT_LIMIT"])
df[df["CREDIT_LIMIT"].isna()]
df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

num_cols = df.select_dtypes(include=["int64", "float64"]).columns

for col in num_cols:
    skew = df[col].skew()
    sns.histplot(df[col], bins=30, kde=True)
    plt.title(f"{col} | Skewness = {skew:.2f}")
    plt.xlabel(col)
    plt.ylabel("Fréquence")
    plt.show()

In [ ]:
skewness = df[num_cols].skew()
print("Variables fortement asymetriques :")
print(skewness[skewness.abs() > 1])

In [ ]:
# corr_matrix = num_cols.corr()
# sns.heatmap(
#     corr_matrix,
#     annot=True,
#     fmt=".2f",
# )
# plt.title("Matrice de corrélation")
# plt.tight_layout()
# plt.show()

In [ ]:
for col in num_cols:
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot de {col}")
    plt.xlabel(col)
    plt.show()

In [ ]:
outliers_in_all = []
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) |(df[col] > upper)]

    outliers_in_all.append({"Variable": col,
        "Nombre d'outliers": len(outliers),
        "Pourcentage": round(len(outliers) / len(df) * 100, 2)
    })
outliers_df = pd.DataFrame(outliers_in_all)
outliers_df

In [ ]:
sns.barplot(
    data=outliers_df,
    x="Variable",
    y="Nombre d'outliers"
)
plt.xticks(rotation=50, ha="right")
plt.title("Nombre d'outliers par variabl")
plt.tight_layout()
plt.show()

In [ ]:
df_clean = df.copy()
df_clean = df_clean.drop_duplicates()
colonnes_inutiles = ["CUST_ID"]
df_clean = df_clean.drop(columns=colonnes_inutiles)
df_clean.to_csv("df_clean.csv",index=False)

In [ ]:
df_clean.columns

In [ ]:
df_prepare_clustering = df_clean.copy()
features = ["BALANCE", "PURCHASES", "ONEOFF_PURCHASES", "INSTALLMENTS_PURCHASES", "CASH_ADVANCE", "CREDIT_LIMIT", "PAYMENTS"]
for col in features:
    if (df_prepare_clustering[col] <0).any():
        print(f"{col} contains zero or negative values")

In [ ]:
import numpy as np
skew_avant = df_prepare_clustering[features].skew()
df_prepare_clustering[features] = np.log1p(df_prepare_clustering[features])
skew_apres = df_prepare_clustering[features].skew()

tableau_skewness = pd.DataFrame({"Skewness avant log": skew_avant, "Skewness apres log": skew_apres})
tableau_skewness["Amelioration"] = (tableau_skewness["Skewness avant log"].abs()- tableau_skewness["Skewness apres log"].abs())
tableau_skewness

In [ ]:

for col in features:
    sns.histplot(
        df_prepare_clustering[col],
        bins=35,
        kde=True,
    )

    plt.title(f"{col} after log1p") 
    plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_prepare_clustering[features])
df_scaled = pd.DataFrame(X_scaled, columns=features,index=df_prepare_clustering.index)
df_scaled

In [ ]:
pca_complete = PCA()
X_pca_complete = pca_complete.fit_transform(df_scaled)

variance_expliquee = pca_complete.explained_variance_ratio_
variance_cumulee = np.cumsum(variance_expliquee)

tableau_pca = pd.DataFrame({
    "Composante": [f"PC{i + 1}" for i in range(len(variance_expliquee))],
    "Variance expliquee": variance_expliquee * 100,
    "Variance cumulee": variance_cumulee * 100
})

tableau_pca.round(2)

In [ ]:
seuil = 0.80
n_composantes_80 = np.argmax(variance_cumulee >= seuil) + 1
print("Nombre de composantes :", n_composantes_80)
print(variance_cumulee[n_composantes_80 - 1].round(2)*100)

In [ ]:
composantes = np.arange(1, len(variance_cumulee) + 1)
plt.plot(
    composantes,
    variance_cumulee * 100,
    marker=".",
)

plt.axhline(
    y=80,
)
plt.xlabel("Nombre de composantes principales")
plt.ylabel("Variance cumulee %")
plt.title("PCA variance expliquee cumulee")
plt.show()

In [ ]:
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

resultats_kmeans = []
for n_composantes in range(2, 6):
    X_pca = X_pca_complete[:, :n_composantes]
    for k in range(2, 8):
        modele = KMeans(
            n_clusters=k,
            random_state=35,
            n_init=10
        )
        labels = modele.fit_predict(X_pca)
        tailles = pd.Series(labels).value_counts(normalize=True)
        silhouette = silhouette_score(
            X_pca,
            labels,
            random_state=35
        )
        resultats_kmeans.append({
            "composantes": n_composantes,
            "k": k,
            "inertie": modele.inertia_,
            "silhouette": silhouette,
            "cluster_min_pouce": tailles.min() * 100,
            "cluster_max_pouce": tailles.max() * 100,
            "equilibre": (tailles.min() >= 0.10 and tailles.max() <= 0.90)
        })
df_kmeans = pd.DataFrame(resultats_kmeans)
df_kmeans.sort_values("silhouette", ascending=False).round(4)


In [ ]:
heatmap_data = df_kmeans.pivot(
    index="composantes",
    columns="k",
    values="silhouette"
)

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".3f",
)
plt.title("Kmeans : silhouette selon PCA et k")
plt.xlabel("Nombre de clusters k")
plt.ylabel("Nombre de composantes PCA")
plt.show()

In [ ]:
meilleure_config = df_kmeans.sort_values("silhouette", ascending=False).iloc[0]
n_km = int(meilleure_config["composantes"])
k_km = int(meilleure_config["k"])
print("Composantes retenues :", n_km)
print("Nombre de clusters :", k_km)

In [ ]:
# X_km = X_pca_complete[:, :n_km]
pca = PCA(2)
X_km = pca.fit_transform(X_scaled)

modele_kmeans = KMeans(
    n_clusters=k_km,
    random_state=35,
    n_init=10
)

labels_kmeans = modele_kmeans.fit_predict(X_km)
courbe = df_kmeans[df_kmeans["composantes"] == n_km].sort_values("k")

sns.scatterplot(
    x = courbe["k"],
    y = courbe["inertie"],
    marker="o"
)
sns.lineplot(x=courbe["k"],y=courbe["inertie"])

plt.title("Methode Elbow")
plt.xlabel("Nombre de clusters k")
plt.ylabel("Inertie")
plt.show()


sns.scatterplot(
    x = courbe["k"],
    y = courbe["silhouette"],
    marker="o"
)
sns.lineplot(x=courbe["k"],y=courbe["silhouette"])

plt.title("Score de silhouette")
plt.xlabel("Nombre de clusters k")
plt.ylabel("Silhouette")
plt.show()

In [ ]:
resultats_dbscan = []
eps_values = [0.2, 0.3, 0.4, 0.5, 0.6]
min_samples_values = [3, 5, 7, 10]

for n_composantes in range(2, 6):
    X_pca = X_pca_complete[:, :n_composantes]
    for eps in eps_values:
        for min_samples in min_samples_values:
            modele = DBSCAN(
                eps=eps,
                min_samples=min_samples
            )
            labels = modele.fit_predict(X_pca)
            masque = labels != -1
            labels_sans_bruit = labels[masque]
            X_sans_bruit = X_pca[masque]
            n_clusters = len(np.unique(labels_sans_bruit))
            proportion_bruit = (~masque).mean() * 100
            if (n_clusters >= 2 and len(labels_sans_bruit) > n_clusters):
                silhouette = silhouette_score(
                    X_sans_bruit,
                    labels_sans_bruit,
                    random_state=35
                )
            resultats_dbscan.append({
                "composantes": n_composantes,
                "eps": eps,
                "min_samples": min_samples,
                "n_clusters": n_clusters,
                "bruit_%": proportion_bruit,
                "silhouette": silhouette
            })
df_dbscan = pd.DataFrame(resultats_dbscan)
df_dbscan = df_dbscan.sort_values("silhouette",ascending=False)
df_dbscan